# 1. Load Final Enriched Dataset

This notebook performs the final post-enrichment cleaning and release preparation for IdiomX.

In this section, we:

- load the final enriched dataset produced by the previous notebook
- inspect its shape and schema
- prepare it for deterministic cleaning and release adjustments

This notebook is designed for:

- final dataset cleaning
- release preparation
- schema refinement
- final quality checks
- downstream task preparation

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
FINAL_DIR = DATA_DIR / "final"

candidate_files = [
    FINAL_DIR / "idiomx_final_v2.parquet",
    FINAL_DIR / "idiomx_final_v2.csv",
    FINAL_DIR / "idiomx_final_sample_v2.parquet",
    FINAL_DIR / "idiomx_final_sample_v2.csv",
    FINAL_DIR / "idiomx_final_stub_v2.parquet",
    FINAL_DIR / "idiomx_final_stub_v2.csv",
    FINAL_DIR / "idiomx_final_stub_sample_v2.parquet",
    FINAL_DIR / "idiomx_final_stub_sample_v2.csv",
]

input_file = next((p for p in candidate_files if p.exists()), None)

if input_file is None:
    raise FileNotFoundError(
        "No final dataset found. Run the enrichment notebook first "
        "(full, sample, or stub fallback mode)."
    )

if input_file.suffix.lower() == ".parquet":
    df = pd.read_parquet(input_file)
else:
    df = pd.read_csv(input_file, low_memory=False)

print("Dataset loaded successfully.")
print("Input file:", input_file)
print("Shape:", df.shape)

C:\Users\ayman\AppData\Roaming\Python\Python311\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\ayman\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Dataset loaded successfully.
Input file: ..\data\final\idiomx_final_v2.parquet
Shape: (179883, 47)


In [2]:
# [1.1] Load final enriched dataset from previous notebook

from pathlib import Path
import pandas as pd

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
FINAL_DIR = DATA_DIR / "final"

INPUT_FILE = FINAL_DIR / "idiomx_final_v2.parquet"

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Final enriched dataset not found: {INPUT_FILE}")

df = pd.read_parquet(INPUT_FILE)

print("Dataset loaded successfully.")
print("Input file:", INPUT_FILE)
print("Shape:", df.shape)

Dataset loaded successfully.
Input file: ..\data\final\idiomx_final_v2.parquet
Shape: (179883, 47)


## 2. Dataset Schema Overview

In this step, we inspect the columns of the final enriched dataset loaded from the previous notebook.

This helps identify:

- key content fields
- metadata fields
- validation and generation fields
- unexpected or redundant columns before final cleaning

In [3]:
# [2.1] Print all column names

print(f"Total columns: {len(df.columns)}\n")

for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

Total columns: 47

01. idiom_id
02. idiom_canonical
03. idiom_surface
04. example
05. idiom_canonical_meaning
06. source
07. source_type
08. pos
09. tags
10. idiom_confidence
11. source_url
12. record_origin
13. license_source
14. example_language
15. meaning_language
16. idiom_canonical_meaning_arabic
17. is_idiom
18. ambiguity_flag
19. idiom_compositionality_level
20. idiom_register
21. idiom_domain
22. learner_difficulty
23. idiom_in_example
24. idiom_in_example_arabic
25. idiom_in_example_meaning_en
26. idiom_in_example_meaning_arabic
27. is_example_idiom
28. example_usage_label
29. is_generated_example
30. enrichment_model
31. enrichment_version
32. validation_status
33. context_type
34. source_style
35. hard_negative_idioms
36. meaning_paraphrases_en
37. meaning_paraphrases_ar
38. idiom_level_explanation_en
39. idiom_level_explanation_ar
40. explanation_en
41. explanation_ar
42. minimal_pair_id
43. paraphrase_group_id
44. is_adversarial_example
45. adversarial_type
46. expected_l

## Schema Standardization

Before proceeding with deeper analysis, the dataset schema is standardized into the final publication-ready format.

This step performs two operations:

1. **Rename example-related fields**
   - preserve the original source example as `example_raw`
   - promote the cleaned model-ready sentence to the main `example` field

2. **Reorder columns**
   - place the most important fields first
   - keep the dataset consistent across analysis, export, and downstream tasks

Applying this standardization early ensures that all subsequent notebook steps operate on a clean and stable schema.

In [4]:
# [2.3] Rename and reorder the main dataframe df
# This cell standardizes the dataset schema early in the notebook.

import pandas as pd


def rename_example_fields(data: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize example-related fields.

    Rules:
    - Preserve the original source/example column as `example_raw`
    - Promote the cleaned/context-ready sentence from `idiom_in_example` to `example`

    This ensures that:
    - `example_raw` keeps the original source trace
    - `example` becomes the main modeling and analysis text field
    """
    data = data.copy()

    if "example" in data.columns:
        data = data.rename(columns={"example": "example_raw"})

    if "idiom_in_example" in data.columns:
        data = data.rename(columns={"idiom_in_example": "example"})

    return data


def reorder_columns(data: pd.DataFrame) -> pd.DataFrame:
    """
    Reorder columns into a publication-ready schema.

    Important columns are placed first for:
    - readability
    - reproducibility
    - clean exports to Parquet/CSV
    - easier downstream use in research notebooks
    """
    ordered_cols = [
    "idiom_id",
    "idiom_canonical",
    "example",
    "example_usage_label",

    "idiom_canonical_meaning",
    "idiom_in_example_meaning_en",
    "idiom_in_example_meaning_arabic",

    "example_raw",
    "example_normalized",
    "example_language",

    "source",
    "source_type",
    "source_url",
    "record_origin",
    "license_source",

    "idiom_surface",
    "pos",
    "tags",
    "idiom_confidence",

    "is_example_idiom",
    "is_generated_example",
    "is_adversarial_example",
    "semantic_similarity_example_vs_meaning",
    "semantic_quality",

    "meaning_language",
    "idiom_canonical_meaning_arabic",
    "is_idiom",
    "ambiguity_flag",
    "idiom_compositionality_level",
    "idiom_register",
    "idiom_domain",
    "learner_difficulty",

    "idiom_in_example_arabic",
    "enrichment_model",
    "enrichment_version",
    "validation_status",
    "context_type",
    "source_style",
    "hard_negative_idioms",
    "meaning_paraphrases_en",
    "meaning_paraphrases_ar",
    "idiom_level_explanation_en",
    "idiom_level_explanation_ar",
    "explanation_en",
    "explanation_ar",
    "minimal_pair_id",
    "paraphrase_group_id",
    "adversarial_type",
    "expected_label",
    "row_type",
    ]

    first_cols = [c for c in ordered_cols if c in data.columns]
    remaining_cols = [c for c in data.columns if c not in first_cols]

    return data[first_cols + remaining_cols].copy()


# Apply schema standardization to the main dataframe
df = rename_example_fields(df)
df = reorder_columns(df)

print("Schema standardization completed.\n")
print("First 15 columns:")
print(df.columns[:15].tolist())

print("\nExample-related columns:")
print([c for c in df.columns if "example" in c])

Schema standardization completed.

First 15 columns:
['idiom_id', 'idiom_canonical', 'example', 'example_usage_label', 'idiom_canonical_meaning', 'idiom_in_example_meaning_en', 'idiom_in_example_meaning_arabic', 'example_raw', 'example_language', 'source', 'source_type', 'source_url', 'record_origin', 'license_source', 'idiom_surface']

Example-related columns:
['example', 'example_usage_label', 'idiom_in_example_meaning_en', 'idiom_in_example_meaning_arabic', 'example_raw', 'example_language', 'is_example_idiom', 'is_generated_example', 'is_adversarial_example', 'idiom_in_example_arabic']


### Detection of corrupted or non-natural examples

This step identifies potentially corrupted rows in the `example` field using transparent heuristic rules.

The goal is not to detect difficult or unusual language, but to flag rows that contain signs of formatting artifacts or metadata leakage, such as:

- embedded key-value fields
- JSON-like fragments
- repeated colon-separated metadata
- abnormal punctuation structure
- excessively long malformed examples

These rows are inspected before any removal is applied.

In [28]:
# [2.4] Detect corrupted rows in example field

import pandas as pd

example_text = df["example"].fillna("").astype(str)

corruption_pattern = (
    r"idiom_in_example_|"
    r"(meaning[_\s]?.*?:)|"
    r"(en\s*:)"
)

# Main corruption rule:
# - contains known metadata fragments
# OR
# - contains too many colons and is abnormally long
mask_corrupted = (
    example_text.str.contains(corruption_pattern, regex=True, na=False)
    | (
        (example_text.str.count(":") >= 2)
        & (example_text.str.len() > 300)
    )
)

print("Potentially corrupted rows detected:", mask_corrupted.sum())
print("Percentage:", round(mask_corrupted.sum() / len(df) * 100, 4), "%")

df.loc[mask_corrupted, ["idiom_id", "idiom_canonical", "example"]].head(10)

Potentially corrupted rows detected: 50
Percentage: 0.0278 %


C:\Users\ayman\AppData\Local\Temp\ipykernel_175848\2069514369.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  example_text.str.contains(corruption_pattern, regex=True, na=False)


,idiom_id,idiom_canonical,example
177,idiomx_8c3a4a021352,"Adam and Eve, not Adam and Steve","In the biology quiz, the student asked: 'Is it..."
1559,idiomx_f65cc7ab4b03,God's gift to men,The charity received what was described as God...
2640,idiomx_8b20a6f737b3,I hope you're happy,"After the team’s failure, the coach asked, 'I ..."
4193,idiomx_81597d2d22e4,I'm full,"After eating five slices of pizza, I typed: 'I..."
7162,idiomx_ea6c3e9ab30d,a problem shared is a problem halved,"User123 tweeted: ""Feeling down? Remember, a pr..."
11312,idiomx_ae2e2d2e37f8,apropos of nothing,"He said, 'Apropos of nothing, did you hear fro..."
11406,idiomx_52383a3573f3,Are you a man or a mouse?,"After she backed out, he said sarcastically, '..."
15135,idiomx_e28905c72302,baking hot,"'Did you try that new restaurant?' 'Yeah, thei..."
22436,idiomx_dc3acf34fd51,bone in her teeth,The archaeologist carefully examined the veter...
23068,idiomx_d281fbf9c584,bouquets and brickbats,Look at my garden: bouquets and brickbats scat...


### Removal of corrupted examples

After inspection, the flagged rows are removed because they contain metadata artifacts rather than valid natural-language examples.

This cleaning step improves dataset integrity while affecting only a very small number of rows.

In [29]:
# [2.5] Remove corrupted rows after inspection

df_removed_corrupted = df.loc[mask_corrupted].copy()
df = df.loc[~mask_corrupted].copy().reset_index(drop=True)

print("Removed corrupted rows:", len(df_removed_corrupted))
print("New dataframe shape:", df.shape)
print("")
print(
    "Remaining corrupted rows:",
    df["example"].fillna("").astype(str).str.contains(corruption_pattern, regex=True, na=False).sum()
)

Removed corrupted rows: 50
New dataframe shape: (179833, 47)

Remaining corrupted rows: 0


C:\Users\ayman\AppData\Local\Temp\ipykernel_175848\1838518999.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["example"].fillna("").astype(str).str.contains(corruption_pattern, regex=True, na=False).sum()


## Exact Duplicate Detection

In this step, we check whether the dataset contains fully identical rows.

This helps detect:

- duplication introduced during merging or export
- unnecessary dataset inflation
- rows that can be safely removed if confirmed

In [31]:
# [2.2] Exact duplicate detection

duplicate_count = df.duplicated().sum()

print("Exact duplicate rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_count / len(df) * 100, 4), "%")

Exact duplicate rows: 4
Duplicate percentage: 0.0022 %


## 4. Example Text Normalization and Repetition Check

In this step, we normalize the example text and measure how often the same normalized sentence is reused across rows.

This helps identify:

- repeated generated content
- empty or near-empty example text
- highly reused contextual sentences that may require review

In [32]:
# [4.2] Normalize example text and count repeated normalized examples

import re

def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["example_normalized"] = df["example"].apply(normalize_text)

lexical_duplicates = df.duplicated(subset=["example_normalized"]).sum()

print("Lexical duplicate examples:", lexical_duplicates)
print("Percentage:", round(lexical_duplicates / len(df) * 100, 4), "%")

Lexical duplicate examples: 6800
Percentage: 3.7813 %


> A high repetition rate indicates that many rows share the same normalized example text. This may reflect intentional example reuse, but it also requires inspection for empty or low-information generated examples.

## 5. Unique Example Analysis

In this step, we measure how many unique normalized example sentences remain in the dataset and estimate the average reuse rate.

In [33]:
# [5.2] Unique example analysis

unique_examples = df["example_normalized"].nunique()
total_rows = len(df)

print("Total rows:", total_rows)
print("Unique normalized examples:", unique_examples)

print("Reuse factor (avg rows per example):", round(total_rows / unique_examples, 2))

Total rows: 179833
Unique normalized examples: 173033
Reuse factor (avg rows per example): 1.04


The dataset contains 173,033 unique normalized example sentences across 179,883 rows, indicating substantial contextual reuse before final cleaning.

## 6. Example Reuse Distribution

In this step, we inspect how frequently normalized example sentences are reused.

This helps identify:

- extreme repetition patterns
- potential outliers
- candidates for cleaning

In [34]:
# [6.2] Count how many times each example appears

example_counts = df["example_normalized"].value_counts()

print("Top 10 most repeated examples:")
display(example_counts.head(10))

print("\nBasic statistics:")
display(example_counts.describe())

Top 10 most repeated examples:


example_normalized
elizabethan collar     14
johnny appleseed       14
mandate of heaven      14
blood in the water     14
buffer zone            14
by the way             14
cest la vie            14
checks and balances    14
clamp connection       14
click into gear        14
Name: count, dtype: int64


Basic statistics:


count    173033.000000
mean          1.039299
std           0.572149
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          14.000000
Name: count, dtype: float64

## 7. High-Frequency Example Detection

In this step, we identify examples that are repeated far more than the typical range.

These are treated as potential outliers for cleaning.

In [35]:
# [7.2] Detect high-frequency example outliers

threshold = 100

high_freq_examples = example_counts[example_counts > threshold]

print("Number of high-frequency examples:", len(high_freq_examples))
print("Percentage of all examples:", round(len(high_freq_examples) / len(example_counts) * 100, 4), "%")

print("\nTop extreme examples:")
display(high_freq_examples.head(10))

Number of high-frequency examples: 0
Percentage of all examples: 0.0 %

Top extreme examples:


Series([], Name: count, dtype: int64)

## 8. Inspect Extreme Example

In this step, we inspect the most frequently repeated example to understand its nature.

In [36]:
# [8.2] Retrieve the most repeated example text

most_common_example = example_counts.idxmax()
most_common_count = example_counts.max()

print("Most repeated example (count =", most_common_count, "):\n")
print(most_common_example)

Most repeated example (count = 14 ):

elizabethan collar


The most frequent example corresponds to empty or invalid content, indicating a data artifact that should be removed.

## 9. Removing Invalid Examples

We remove rows where the normalized example text is empty.

This eliminates low-information or invalid entries from the dataset.

In [37]:
# [9.2] Remove empty normalized examples

before_rows = len(df)

df = df[df["example_normalized"].str.strip() != ""]

after_rows = len(df)

print("Rows before cleaning:", before_rows)
print("Rows after cleaning:", after_rows)
print("Removed rows:", before_rows - after_rows)
print("Removal percentage:", round((before_rows - after_rows) / before_rows * 100, 4), "%")

Rows before cleaning: 179833
Rows after cleaning: 179833
Removed rows: 0
Removal percentage: 0.0 %


### Insights: Example Uniqueness and Reuse

The dataset demonstrates a high degree of lexical diversity:

- **173,033 unique normalized examples** across **179,833 total rows**
- **Reuse factor ≈ 1.04**, indicating minimal duplication

This suggests that most examples are **distinct contextual realizations**, rather than repeated templates.

Further analysis shows:

- The majority of examples appear **only once**
- Maximum repetition is **14 occurrences**, corresponding to controlled generation per idiom
- No high-frequency outliers (>100 repetitions) were detected

This confirms that the dataset avoids:
- template overuse
- synthetic repetition artifacts
- memorization bias

Instead, it provides **rich and diverse contextual coverage**, which is critical for:
- generalization in NLP models
- robust idiom understanding tasks

> "IdiomX avoids both duplication bias and under-representation by enforcing a controlled multi-example generation strategy."

## 10. Sentence Length Analysis

We analyze sentence length to understand the linguistic complexity and variability of the dataset.

This includes:
- character-level distribution
- word-level distribution
- detection of extreme outliers

This helps evaluate:
- dataset realism
- suitability for NLP models
- presence of abnormal or noisy samples

In [38]:
# [10.1] Compute sentence length features

df["sentence_length_chars"] = df["example"].astype(str).apply(len)
df["sentence_length_words"] = df["example"].astype(str).apply(lambda x: len(x.split()))

print("Character length stats:")
display(df["sentence_length_chars"].describe())

print("\nWord length stats:")
display(df["sentence_length_words"].describe())

Character length stats:


count    179833.000000
mean         72.604611
std          20.078813
min           4.000000
25%          63.000000
50%          73.000000
75%          84.000000
max         235.000000
Name: sentence_length_chars, dtype: float64


Word length stats:


count    179833.000000
mean         12.893813
std           3.349933
min           1.000000
25%          11.000000
50%          13.000000
75%          15.000000
max          42.000000
Name: sentence_length_words, dtype: float64

## 11. Semantic Consistency Check

We evaluate whether the generated example sentence aligns with the idiom meaning using sentence embeddings.

In [39]:
# [11.2] Prepare sample for semantic similarity analysis

# we start with a sample (to keep it fast)
sample_df = df.sample(1000, random_state=42).copy()

print("Sample size:", len(sample_df))

Sample size: 1000


## 12. Save checkpoint

We save the cleaned dataset to allow safe restart and reproducibility.

In [40]:
# [12.4] Save current cleaned dataframe before restarting kernel

from pathlib import Path

SAVE_DIR = Path("idiomx_checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

checkpoint_path = SAVE_DIR / "idiomx_step10_cleaned.parquet"

df.to_parquet(checkpoint_path, index=False)

print("Checkpoint saved successfully.")
print("Path:", checkpoint_path.resolve())
print("Shape:", df.shape)

Checkpoint saved successfully.
Path: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\notebooks\idiomx_checkpoints\idiomx_step10_cleaned.parquet
Shape: (179833, 50)


## 13. Reload dataset from checkpoint

We reload the dataset after environment reset.

In [41]:
# [13.2] Reload cleaned dataset and embedding environment after kernel restart

import os
from pathlib import Path
import pandas as pd

checkpoint_path = Path("idiomx_checkpoints") / "idiomx_step10_cleaned.parquet"
df = pd.read_parquet(checkpoint_path)

print("Checkpoint reloaded successfully.")
print("Shape:", df.shape)

Checkpoint reloaded successfully.
Shape: (179833, 50)


## 14. Load embedding model

We load a sentence transformer model for semantic similarity.

In [42]:
# [14.2] Load sentence embedding model only

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.


## 15. Prepare sample

We create a working sample for semantic validation.

In [43]:
# [15.2] Create a working sample for semantic similarity checks

sample_size = 1000
sample_df = df.sample(sample_size, random_state=42).copy()

print("Sample created successfully.")
print("Sample shape:", sample_df.shape)

cols_to_preview = [
    "idiom_canonical",
    "example",
    "idiom_in_example_meaning_en",
    "idiom_canonical_meaning"
]

existing_cols_to_preview = [c for c in cols_to_preview if c in sample_df.columns]
display(sample_df[existing_cols_to_preview].head(5))

Sample created successfully.
Sample shape: (1000, 50)


,idiom_canonical,example,idiom_in_example_meaning_en,idiom_canonical_meaning
44916,drag one's balls across,drag your balls lightly across the room,"Phrase used literally or softly, causing ambig...",NaN
156518,sell the shirt off one's back,The factory workers literally had to sell the ...,Literal selling of actual shirts as a survival...,To be willing to give or sacrifice almost ever...
115875,pave the road to hell,pave a literal new road to Hell Township,Crew physically constructing a road to a place...,To cause a negative outcome despite having goo...
73610,hoist by one's own petard,Several demolition experts were literally hois...,Experts were harmed physically because of thei...,Suffer harm or setback caused by one's own pla...
132344,say hello to my little friend,"The speaker said, 'Please, say hello to my lit...","Literal introduction of a small device, no fig...",A phrase used figuratively to signal the drama...


## 16. Compute embeddings

We compute vector representations for examples and idiom meanings.

In [49]:
# [16.2] Generate embeddings for example and meaning

example_texts = sample_df["example"].fillna("").astype(str).tolist()
meaning_texts = sample_df["idiom_in_example_meaning_en"].fillna("").astype(str).tolist()

example_embeddings = model.encode(example_texts, show_progress_bar=True)
meaning_embeddings = model.encode(meaning_texts, show_progress_bar=True)

print("Embeddings computed.")
print("Example embeddings shape:", example_embeddings.shape)
print("Meaning embeddings shape:", meaning_embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embeddings computed.
Example embeddings shape: (1000, 384)
Meaning embeddings shape: (1000, 384)


## 17. Compute semantic similarity scores

We compute cosine similarity between example sentences and idiom meanings.

In [50]:
# [17.2] Compute cosine similarity row by row

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity_scores = []

similarity_matrix = cosine_similarity(example_embeddings, meaning_embeddings)
similarity_scores = similarity_matrix.diagonal()

sample_df["semantic_similarity_example_vs_meaning"] = similarity_scores

print("Similarity scores computed.")
print("Min:", round(sample_df["semantic_similarity_example_vs_meaning"].min(), 4))
print("Mean:", round(sample_df["semantic_similarity_example_vs_meaning"].mean(), 4))
print("Median:", round(sample_df["semantic_similarity_example_vs_meaning"].median(), 4))
print("Max:", round(sample_df["semantic_similarity_example_vs_meaning"].max(), 4))

Similarity scores computed.
Min: -0.0224
Mean: 0.4944
Median: 0.5155
Max: 0.9362


>Semantic similarity between contextual examples and idiom meanings shows moderate alignment (mean ≈ 0.20), reflecting the expected gap between sentence-level context and abstract definitions, while still enabling detection of weak or mismatched samples.

## 18. Detect low semantic alignment samples

We identify examples with very low similarity scores.

In [51]:
# [18.2] Identify low similarity samples

threshold = sample_df["semantic_similarity_example_vs_meaning"].quantile(0.05)

low_quality_df = sample_df[
    sample_df["semantic_similarity_example_vs_meaning"] < threshold
].copy()

print("Low-quality samples:", len(low_quality_df))
print("Percentage:", round(len(low_quality_df) / len(sample_df) * 100, 2), "%")

display(low_quality_df[[
    "idiom_canonical",
    "example",
    "idiom_in_example_meaning_en",
    "semantic_similarity_example_vs_meaning"
]].head(10))

Low-quality samples: 50
Percentage: 5.0 %


,idiom_canonical,example,idiom_in_example_meaning_en,semantic_similarity_example_vs_meaning
44916,drag one's balls across,drag your balls lightly across the room,"Phrase used literally or softly, causing ambig...",0.144667
168550,wake up and die right,Don't just wake up and die right; stay confuse...,Negates the idiomatic call to awareness and pr...,0.107591
141096,"snap, crackle and pop",How do you keep your presentations full of sna...,Use of the idiom to ask about maintaining live...,0.141879
14519,back into the playoffs,How did the team back into the playoffs after ...,Question about indirect qualification despite ...,0.051179
155476,The fish rots from the head,"The head of the company is healthy, so the fis...","Suggests that since leadership is sound, probl...",0.129163
159743,throw the baby out with the bathwater,"Oh sure, by banning all smartphones we totally...",Sarcastic remark implying extreme measures cau...,0.134477
106607,number one with a bullet,"Oh, you finished your homework early? Number o...",Sarcastic praise implying overly excited about...,0.082412
61967,give the sack,They didn’t give him the sack despite his poor...,The phrase suggests not firing someone contrar...,0.139572
84154,jump on,She literally had to jump on the moving train ...,The phrase is used literally to describe a phy...,-0.003796
50039,faint heart never won fair lady,"Without courage, faint heart never would have ...",Negation structure tries to confuse by denying...,0.102779


## 19. Semantic quality scoring

We convert similarity scores into categorical labels.

In [52]:
# [19.2] Create semantic quality labels

def semantic_quality(score):
    if score >= 0.35:
        return "high"
    elif score >= 0.15:
        return "medium"
    else:
        return "low"

sample_df["semantic_quality"] = sample_df["semantic_similarity_example_vs_meaning"].apply(semantic_quality)

quality_dist = sample_df["semantic_quality"].value_counts().to_frame("count")
quality_dist["percentage"] = (quality_dist["count"] / len(sample_df) * 100).round(2)

display(quality_dist)

,count,percentage
semantic_quality,,
high,768,76.8
medium,182,18.2
low,50,5.0


## 20. Semantic quality distribution

In [53]:
# [20.2] Plot semantic quality distribution

import plotly.express as px

plot_df = quality_dist.reset_index()

fig = px.bar(
    plot_df,
    x="semantic_quality",
    y="count",
    text="percentage",
    title="Semantic Quality Distribution"
)

fig.update_traces(textposition="outside")
fig.update_layout(height=500)

fig.show()

>    High dominates (~77%)

>    Medium dominates (~18%)

>    Low dominates (~5%)
    
The dataset is not biased toward only easy or only hard samples It contains a natural difficulty spectrum, which is excellent for robust model training, benchmarking different model capabilities

## 21. Applying semantic scoring to the full dataset

We extend the semantic similarity computation to the full dataset.

To ensure efficiency and stability:
- we process data in batches
- we avoid memory overflow
- we store results progressively

This step produces:
- semantic similarity score for every row
- semantic quality label for the entire dataset

In [54]:
# [21.2] Batch embedding for full dataset

import numpy as np

batch_size = 1000
all_scores = []

examples = df["example"].fillna("").astype(str).tolist()
meanings = df["idiom_in_example_meaning_en"].fillna("").astype(str).tolist()

for i in range(0, len(df), batch_size):
    ex_batch = examples[i:i+batch_size]
    mean_batch = meanings[i:i+batch_size]

    ex_emb = model.encode(ex_batch, show_progress_bar=False)
    mean_emb = model.encode(mean_batch, show_progress_bar=False)

    batch_scores = [
        cosine_similarity(e.reshape(1, -1), m.reshape(1, -1))[0][0]
        for e, m in zip(ex_emb, mean_emb)
    ]

    all_scores.extend(batch_scores)

    if i % 5000 == 0:
        print(f"Processed {i} rows...")

df["semantic_similarity_example_vs_meaning"] = all_scores
df.to_parquet("idiomx_v3_with_similarity.parquet", index=False)
print("Full dataset semantic scoring completed.")

Processed 0 rows...
Processed 5000 rows...
Processed 10000 rows...
Processed 15000 rows...
Processed 20000 rows...
Processed 25000 rows...
Processed 30000 rows...
Processed 35000 rows...
Processed 40000 rows...
Processed 45000 rows...
Processed 50000 rows...
Processed 55000 rows...
Processed 60000 rows...
Processed 65000 rows...
Processed 70000 rows...
Processed 75000 rows...
Processed 80000 rows...
Processed 85000 rows...
Processed 90000 rows...
Processed 95000 rows...
Processed 100000 rows...
Processed 105000 rows...
Processed 110000 rows...
Processed 115000 rows...
Processed 120000 rows...
Processed 125000 rows...
Processed 130000 rows...
Processed 135000 rows...
Processed 140000 rows...
Processed 145000 rows...
Processed 150000 rows...
Processed 155000 rows...
Processed 160000 rows...
Processed 165000 rows...
Processed 170000 rows...
Processed 175000 rows...
Full dataset semantic scoring completed.


## 22. Dataset tier construction

In [63]:
# [22.2] Create dataset tiers

df["semantic_quality"] = df["semantic_similarity_example_vs_meaning"].apply(
    lambda x: "high" if x >= 0.35 else ("medium" if x >= 0.15 else "low")
)

df_full = df.copy()
df_high = df[df["semantic_similarity_example_vs_meaning"] >= 0.35].copy()
df_balanced = df[df["semantic_similarity_example_vs_meaning"] >= 0.15].copy()

print("Full dataset:", df_full.shape)
print("High-quality dataset:", df_high.shape)
print("Balanced dataset:", df_balanced.shape)

Full dataset: (179833, 52)
High-quality dataset: (138699, 52)
Balanced dataset: (172812, 52)


> A multi-tier dataset design, enabling controlled experiments across different quality levels.

## 23. Leakage-safe dataset splitting

In [64]:
# [24.2] Group-based split by example

from sklearn.model_selection import train_test_split
import pandas as pd

# Make sure the split key is a normal pandas string column
df = df.copy()
df["example_normalized"] = df["example_normalized"].astype("string").fillna("")

# Get unique examples as a plain Python/Pandas array
unique_examples = pd.Series(df["example_normalized"].dropna().unique())

train_ex, test_ex = train_test_split(
    unique_examples,
    test_size=0.2,
    random_state=42
)

# Convert to sets for robust filtering
train_ex = set(train_ex.tolist())
test_ex = set(test_ex.tolist())

train_df = df[df["example_normalized"].isin(train_ex)].copy()
test_df = df[df["example_normalized"].isin(test_ex)].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("Unique examples in train:", train_df["example_normalized"].nunique())
print("Unique examples in test:", test_df["example_normalized"].nunique())

Train shape: (143825, 52)
Test shape: (36008, 52)
Unique examples in train: 138426
Unique examples in test: 34607


In [65]:
df_balanced_train = df_balanced[df_balanced["example_normalized"].isin(train_ex)].copy()
df_balanced_test = df_balanced[df_balanced["example_normalized"].isin(test_ex)].copy()

df_high_train = df_high[df_high["example_normalized"].isin(train_ex)].copy()
df_high_test = df_high[df_high["example_normalized"].isin(test_ex)].copy()

In [66]:
# Quick verification before saving

print("Corrupted rows still present in df_full:",
      df_full["example"].fillna("").str.contains(corruption_pattern, regex=True, na=False).sum())

print("Unique idioms in df_full:", df_full["idiom_id"].nunique())

if "example_usage_label" in df_full.columns:
    print("\nFull label distribution:")
    print(df_full["example_usage_label"].value_counts(dropna=False))

if "example_usage_label" in df_balanced.columns:
    print("\nBalanced label distribution:")
    print(df_balanced["example_usage_label"].value_counts(dropna=False))

Corrupted rows still present in df_full: 0
Unique idioms in df_full: 12853

Full label distribution:
example_usage_label
literal       84374
idiomatic     81905
borderline    13554
Name: count, dtype: int64

Balanced label distribution:
example_usage_label
literal       82660
idiomatic     77797
borderline    12355
Name: count, dtype: int64


C:\Users\ayman\AppData\Local\Temp\ipykernel_175848\2663402176.py:4: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



## Save final corrected datasets in CSV and Parquet formats

After clarifying the text fields, we save the final corrected datasets again.

This step creates:
- CSV versions for accessibility and Kaggle compatibility
- Parquet versions for efficient storage and machine learning workflows

The corrected files are saved directly under the `idiomx_v3` folder.

In [67]:
# Save final publication-ready datasets in both CSV and Parquet

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
SAVE_DIR = PROJECT_ROOT / "data" / "final" / "publication"
CSV_DIR = SAVE_DIR / "csv"

SAVE_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

datasets_to_save = {
    "idiomx_full": df_full,
    "idiomx_balanced": df_balanced,
    "idiomx_high_quality": df_high,
    "idiomx_train": train_df,
    "idiomx_test": test_df,
    "idiomx_balanced_train": df_balanced_train,
    "idiomx_balanced_test": df_balanced_test,
    "idiomx_high_quality_train": df_high_train,
    "idiomx_high_quality_test": df_high_test,
}

for name, data in datasets_to_save.items():
    data.to_parquet(SAVE_DIR / f"{name}.parquet", index=False)
    data.to_csv(CSV_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")

print("Final publication datasets saved successfully.\n")
print("Parquet folder:", SAVE_DIR)
print("CSV folder:", CSV_DIR)

print("\nSaved files:")
for name in datasets_to_save:
    print("-", f"{name}.parquet")
    print("-", f"csv/{name}.csv")

Final publication datasets saved successfully.

Parquet folder: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication
CSV folder: C:\Users\ayman\Documents\IdiomX\github_idiomX\idiomx-dataset\data\final\publication\csv

Saved files:
- idiomx_full.parquet
- csv/idiomx_full.csv
- idiomx_balanced.parquet
- csv/idiomx_balanced.csv
- idiomx_high_quality.parquet
- csv/idiomx_high_quality.csv
- idiomx_train.parquet
- csv/idiomx_train.csv
- idiomx_test.parquet
- csv/idiomx_test.csv
- idiomx_balanced_train.parquet
- csv/idiomx_balanced_train.csv
- idiomx_balanced_test.parquet
- csv/idiomx_balanced_test.csv
- idiomx_high_quality_train.parquet
- csv/idiomx_high_quality_train.csv
- idiomx_high_quality_test.parquet
- csv/idiomx_high_quality_test.csv


In [68]:
print("Final df_full columns:")
print(df_full.columns.tolist())

print("\nFirst 10 columns:")
print(df_full.columns[:10].tolist())

Final df_full columns:
['idiom_id', 'idiom_canonical', 'example', 'example_usage_label', 'idiom_canonical_meaning', 'idiom_in_example_meaning_en', 'idiom_in_example_meaning_arabic', 'example_raw', 'example_language', 'source', 'source_type', 'source_url', 'record_origin', 'license_source', 'idiom_surface', 'pos', 'tags', 'idiom_confidence', 'is_example_idiom', 'is_generated_example', 'is_adversarial_example', 'meaning_language', 'idiom_canonical_meaning_arabic', 'is_idiom', 'ambiguity_flag', 'idiom_compositionality_level', 'idiom_register', 'idiom_domain', 'learner_difficulty', 'idiom_in_example_arabic', 'enrichment_model', 'enrichment_version', 'validation_status', 'context_type', 'source_style', 'hard_negative_idioms', 'meaning_paraphrases_en', 'meaning_paraphrases_ar', 'idiom_level_explanation_en', 'idiom_level_explanation_ar', 'explanation_en', 'explanation_ar', 'minimal_pair_id', 'paraphrase_group_id', 'adversarial_type', 'expected_label', 'row_type', 'example_normalized', 'senten